# ADAC Workshop 1 26L: Performance & Computing Models

## Introduction
In this lab, you will compare the performance and computing models of four popular data processing libraries/engines: **Polars, Pandas and DuckDB**. We will include PySpark distributed engine in the introduction, but we will not focus on distributed execution in this lab.

You will explore:
- **Performance**: single-node processing speed, parallel execution, memory usage, and result materialization cost.
- **Scalability**: how performance changes with the number of local threads/cores.
- **Physical layout**: how file format, Parquet layout, row groups, sorting, partitioning, and pruning affect IO.
- **Computing models**: in-memory vs. out-of-core processing, SQL vs. DataFrame APIs, eager vs. lazy execution, and streaming execution vs. streaming output.


## Submission identity

Before starting the assignment, copy this notebook into your fork of the workshop repository and work on that copy.

Fill in the first code cell with:

- your full name,
- a link to this notebook in your forked GitHub repository,
- names or IDs of group members if required by the instructor.

The submitted notebook should be reachable from your fork. Do not submit a notebook that only exists locally.
The notebook is stored in the repository: https://github.com/bdg-tbd/tbd-workshop-1/tree/ADAC26L/notebooks/ . TBD is another WUT course that uses similar notebooks to ADAC, so we use the same repository for both courses. 

In [3]:
# TODO: Fill this in before submitting. Change <your-github-user-or-org> and <branch>
FULL_NAME = "Kacper_Pawlowski"
NOTEBOOK_URL = "https://github.com/KacperskiWDC/tbd-workshop-1_ADAC/blob/master/notebooks/adac_lab1_26L.ipynb"

assert FULL_NAME is not None, "Set FULL_NAME before running the notebook"
assert "<your-github-user-or-org>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"
assert "<branch>" not in NOTEBOOK_URL, "Set NOTEBOOK_URL to your forked repository notebook URL"

## Library/engine capabilities

Use this table as a reference when interpreting your results.

| Library/engine | Query optimizer | Distributed | Arrow-backed | Out-of-core | Parallel local execution | Main APIs |
|---|---|---|---|---|---|---|
| **Pandas 3.0.2** | no | no | default IO returns NumPy-backed data; `dtype_backend="pyarrow"` returns PyArrow-backed nullable dtypes | no | limited | DataFrame, `pd.col` for selected expression-style usage |
| **Polars** | yes | single-node locally; distributed engine is available in Polars Cloud and is outside this local benchmark | yes | yes | yes | DataFrame, lazy expressions, SQL subset |
| **DuckDB** | yes | no | no, but Arrow-compatible (zero-copy interop) | yes | yes | SQL, relational API |
| **PySpark** | yes | yes | no | yes | yes | SQL, DataFrame |

The goal is not to prove that one library is always best. The goal is to identify which library/engine is appropriate for a given data size, query shape, memory limit, physical layout, and deployment model.

Use pandas 3.0.2 in this lab. Two pandas 3.0.2 behaviours matter for the benchmark:
- string columns are no longer inferred as generic `object` dtype by default,
- Copy-on-Write is the only mutation model (in pandas 2.x there could be unintended side effects of mutating DataFrames - in 3.x every object behaves as an independent copy).

In addition, compare two Pandas Parquet-reading variants where possible:  
- default Pandas/NumPy-backed DataFrame: `pd.read_parquet(path)`,
- PyArrow-backed DataFrame: `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`.

Record the pandas version and dtypes in your report.


## Parquet introduction

Image and introduction reference: [Parquet pruning pipeline in DataFusion](https://datafusion.apache.org/blog/2025/03/20/parquet-pruning/)

![Parquet pruning pipeline in DataFusion](https://datafusion.apache.org/blog/images/parquet-pruning/read-parquet.jpg)

## Prerequisites

Install the required libraries in your notebook environment. Pandas 3.0.2 requires Python 3.11 or newer.

Use current Polars API in new code. In particular, use `collect(engine="streaming")` for streaming execution and use sink methods when you want to write streaming output to disk.

For Pandas, benchmark both the default backend and the PyArrow dtype backend for Parquet reads. The PyArrow-backed variant is especially relevant for string-heavy datasets.

Use the cell below to install the dependencies:


In [ ]:
%pip install -U "pandas>=3.0,<3.1" polars duckdb memory_profiler psutil matplotlib seaborn

In [4]:
import gc
import os
import time
import json
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import duckdb
import psutil
from memory_profiler import memory_usage
import pyarrow.parquet as pq
import pyarrow

print("Python:", platform.python_version())
if tuple(map(int, platform.python_version_tuple()[:2])) < (3, 11):
    raise RuntimeError("This notebook requires Python 3.11+ because it uses pandas 3.0.2.")
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)
if tuple(map(int, pd.__version__.split(".")[:2])) < (3, 0):
    raise RuntimeError("Install pandas 3.0+ before running the benchmark.")
print("DuckDB:", duckdb.__version__)
print("CPU logical cores:", psutil.cpu_count(logical=True))
print("RAM GiB:", round(psutil.virtual_memory().total / 2**30, 2))


Python: 3.14.5
Polars: 1.40.1
Pandas: 3.0.3
PyArrow: 24.0.0
DuckDB: 1.5.3
CPU logical cores: 22
RAM GiB: 15.46


## Part 1: Download dataset

During the ADAC Workshop 1, we will use [Microsoft Security Incident Prediction Dataset](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction/data) from Kaggle. First we need to download it from Kaggle, using `kagglehub` library.

In [5]:
import kagglehub
path = kagglehub.dataset_download("Microsoft/microsoft-security-incident-prediction")
path

100%|█████████████████████████████████████████████████| 513M/513M [09:21<00:00, 959kB/s]

Extracting files...


'C:\\Users\\kacpe\\.cache\\kagglehub\\datasets\\Microsoft\\microsoft-security-incident-prediction\\versions\\1'

In [8]:
import os

total_size = 0
for dirpath, dirnames, filenames in os.walk(path):
    for filename in filenames:
        file_path = os.path.join(dirpath, filename)
        total_size += os.path.getsize(file_path)

print("Dataset size GiB:", round(total_size / 2**30, 2))

Dataset size GiB: 3.27


In [6]:
!du -sh {path}

'du' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
!find {path} -ls

FIND: Parameter format not correct


In [9]:
import os
train_path = os.path.join(path, "GUIDE_Train.csv")
test_path = os.path.join(path, "GUIDE_Test.csv")
train_path, test_path

('C:\\Users\\kacpe\\.cache\\kagglehub\\datasets\\Microsoft\\microsoft-security-incident-prediction\\versions\\1\\GUIDE_Train.csv',
 'C:\\Users\\kacpe\\.cache\\kagglehub\\datasets\\Microsoft\\microsoft-security-incident-prediction\\versions\\1\\GUIDE_Test.csv')

### Part 1.2: Exploring the dataset

Next, you need to explore the dataset on [Kaggle incident dataset webpage](https://www.kaggle.com/datasets/Microsoft/microsoft-security-incident-prediction/data).

You can also explore it by performing data exploration analysis in Python below (statistical information about columns, data quantity, making simple visualisations).

In [11]:
# Data exploration analysis (optional)

# Basic dataset exploration
df_info = pl.scan_csv(train_path, infer_schema_length=20_000)

print("Columns:")
print(df_info.collect_schema().names())

print("\nNumber of rows:")
print(df_info.select(pl.len()).collect())

Columns:
['Id', 'OrgId', 'IncidentId', 'AlertId', 'Timestamp', 'DetectorId', 'AlertTitle', 'Category', 'MitreTechniques', 'IncidentGrade', 'ActionGrouped', 'ActionGranular', 'EntityType', 'EvidenceRole', 'DeviceId', 'Sha256', 'IpAddress', 'Url', 'AccountSid', 'AccountUpn', 'AccountObjectId', 'AccountName', 'DeviceName', 'NetworkMessageId', 'EmailClusterId', 'RegistryKey', 'RegistryValueName', 'RegistryValueData', 'ApplicationId', 'ApplicationName', 'OAuthApplicationId', 'ThreatFamily', 'FileName', 'FolderPath', 'ResourceIdName', 'ResourceType', 'Roles', 'OSFamily', 'OSVersion', 'AntispamDirection', 'SuspicionLevel', 'LastVerdict', 'CountryCode', 'State', 'City']

Number of rows:
shape: (1, 1)
┌─────────┐
│ len     │
│ ---     │
│ u32     │
╞═════════╡
│ 9516837 │
└─────────┘


### 1.3: Partitioning the dataset

In order to optimize the performance of the queries, we need to partition the dataset. We will use the `ts_date_bucketed` column to partition the dataset.

In [10]:
import shutil
# --- 1.3 Partitioning the dataset -------------------------------------------------
# Path constants used by the benchmark helpers (Part 2) and by Task 2.5.
WORKDIR = Path("data_adac1"); WORKDIR.mkdir(exist_ok=True)
EVENTS_PATH            = str(WORKDIR / "guide_train.parquet")          # default single-file Parquet
OPTIMIZED_EVENTS_PATH  = str(WORKDIR / "guide_train_sorted.parquet")   # sorted on the filter keys + small row groups
PARTITIONED_EVENTS_DIR = str(WORKDIR / "guide_train_by_date")          # Hive-partitioned by ts_date_bucketed
CSV_EVENTS_PATH        = str(WORKDIR / "guide_q1_slice.csv")           # flat CSV negative baseline (only the query's columns)

# (a) one-off CSV -> single Parquet (the dataset ships as CSV; Task 2 / 2.5 read Parquet)
if not os.path.exists(EVENTS_PATH):
    t0 = time.perf_counter()
    pl.scan_csv(train_path, infer_schema_length=20_000).sink_parquet(EVENTS_PATH, compression="zstd")
    print(f"csv -> parquet in {time.perf_counter() - t0:.1f} s")

con = duckdb.connect(); con.execute("SET enable_progress_bar = false")

# (b) derive ts_date_bucketed: the Timestamp dump is heavily front-tailed -- a few thousand rows are
#     scattered across late-2023 / early-2024, then ~99.5% of the rows fall in ~4 weeks of May-Jun 2024.
#     A raw daily Hive partition would create ~150 near-empty directories, so we keep each "dense" day as
#     its own partition and collapse the oldest ~0.5% of rows into one "_other" partition -> ~30 fairly
#     even partitions. (Rationale, metrics and plots: exploratory-analysis/report.html.)
TAIL_FRACTION = 0.005
date_sql   = 'CAST(substr("Timestamp", 1, 10) AS DATE)'
cutoff     = con.execute(f"SELECT quantile_disc({date_sql}, {TAIL_FRACTION}) FROM read_parquet('{EVENTS_PATH}')").fetchone()[0]
bucket_sql = f"CASE WHEN {date_sql} >= DATE '{cutoff}' THEN CAST({date_sql} AS VARCHAR) ELSE '_other' END"
print(f"ts_date_bucketed: dense-day cutoff = {cutoff} (rows older than this date -> '_other')")

# (c) build the three alternative physical layouts with DuckDB COPY
t0 = time.perf_counter()
shutil.rmtree(PARTITIONED_EVENTS_DIR, ignore_errors=True)
con.execute(f"COPY (SELECT *, {bucket_sql} AS ts_date_bucketed FROM read_parquet('{EVENTS_PATH}')) "
            f"TO '{PARTITIONED_EVENTS_DIR}' (FORMAT PARQUET, COMPRESSION ZSTD, PARTITION_BY (ts_date_bucketed), OVERWRITE_OR_IGNORE TRUE)")
con.execute(f"COPY (SELECT * FROM read_parquet('{EVENTS_PATH}') ORDER BY Category, IncidentGrade, CountryCode) "
            f"TO '{OPTIMIZED_EVENTS_PATH}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000, OVERWRITE_OR_IGNORE TRUE)")
con.execute(f"COPY (SELECT Category, IncidentGrade, CountryCode, IncidentId, MitreTechniques FROM read_parquet('{EVENTS_PATH}')) "
            f"TO '{CSV_EVENTS_PATH}' (FORMAT CSV, HEADER TRUE)")
print(f"built alternative layouts in {time.perf_counter() - t0:.1f} s")

# (d) report sizes + how even the date partitions turned out
pp = con.execute(f"SELECT COUNT(*) n FROM read_parquet('{PARTITIONED_EVENTS_DIR}/**/*.parquet', hive_partitioning=true) GROUP BY ts_date_bucketed").df()["n"]
con.close()
def _dirsize(p): return os.path.getsize(p) if os.path.isfile(p) else sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(p) for f in fs)
def _nfiles(p):  return 1 if os.path.isfile(p) else sum(len(fs) for _, _, fs in os.walk(p))
for name, p in [("default parquet (1 file)", EVENTS_PATH), ("sorted parquet (rg=100k)", OPTIMIZED_EVENTS_PATH),
                ("partitioned parquet (by date bucket)", PARTITIONED_EVENTS_DIR), ("csv slice (5 cols, flat)", CSV_EVENTS_PATH)]:
    print(f"  {name:38s} {_dirsize(p)/2**20:9.1f} MB   {_nfiles(p):3d} file(s)")
print(f"  -> {len(pp)} date partitions; rows/partition  min={pp.min():,}  median={int(pp.median()):,}  max={pp.max():,}")


csv -> parquet in 25.1 s
ts_date_bucketed: dense-day cutoff = 2024-05-20 (rows older than this date -> '_other')
built alternative layouts in 67.6 s
  default parquet (1 file)                   299.2 MB     1 file(s)
  sorted parquet (rg=100k)                   220.7 MB     1 file(s)
  partitioned parquet (by date bucket)       258.4 MB    30 file(s)
  csv slice (5 cols, flat)                   392.7 MB     1 file(s)
  -> 30 date partitions; rows/partition  min=2,506  median=106,337  max=910,606


## Part 2: Measuring performance

You must use one consistent benchmark protocol for all libraries/engines.

Minimum requirements:

1. Run every benchmark at least three times. Five repetitions are recommended.
2. Run `gc.collect()` before each measured repetition to reduce noise from previous Python allocations.
3. Report median runtime, not only one measurement.
4. Record peak memory where possible.
5. Check that results are logically equivalent across libraries/engines.
6. Store your results in a table.
7. Describe any library/engine-specific settings, such as Pandas dtype backend, thread count, Spark local mode, or DuckDB threads.

**Important for memory benchmarks**: notebook kernels keep allocations and library state between cells. Peak-RSS comparisons are often misleading when all variants run in the same process. For Task 4 and any memory-sensitive comparison, prefer running each variant in a fresh process or a small standalone script. If you cannot do that, clearly state this limitation.

You may use the implemented helper below.


In [13]:
import hashlib, threading
import pyarrow.parquet as pq
# --- Part 2: Measuring performance ------------------------------------------------------------------
# Complete benchmark harness -- you do NOT edit this cell. In Tasks 2-5 you call benchmark_case(...) to
# time a query in one library/engine, and show_results() to get the results table. Path constants
# (EVENTS_PATH / PARTITIONED_EVENTS_DIR / OPTIMIZED_EVENTS_PATH / CSV_EVENTS_PATH) come from Part 1.3.
REPEATS = 5  # >= 3 required, 5 recommended (Part 2 protocol); per-call override via benchmark_case(..., repeats=)

BENCHMARK_COLUMNS = [
    "library_engine", "mode", "query_name", "data_format", "layout",
    "rows", "median_time_s", "iqr_time_s", "peak_memory_mb", "input_size_mb",
    "files", "row_groups", "result_check", "notes",
]

def parquet_source(layout):
    if layout == "unpartitioned": return str(EVENTS_PATH)
    if layout == "partitioned":   return str(PARTITIONED_EVENTS_DIR)
    if layout == "optimized":     return str(OPTIMIZED_EVENTS_PATH)
    raise ValueError(layout)

def parquet_glob(layout):
    if layout == "unpartitioned": return str(EVENTS_PATH)
    if layout == "partitioned":   return os.path.join(str(PARTITIONED_EVENTS_DIR), "**", "*.parquet")
    if layout == "optimized":     return str(OPTIMIZED_EVENTS_PATH)
    raise ValueError(layout)

def layout_profile(layout, data_format="parquet"):
    if data_format == "csv":
        return {"files": 1, "row_groups": None, "input_size_mb": round(os.path.getsize(CSV_EVENTS_PATH) / 2**20, 2)}
    path = Path(parquet_source(layout))
    files = sorted(path.rglob("*.parquet")) if path.is_dir() else [path]
    row_groups = sum(pq.ParquetFile(f).num_row_groups for f in files)
    total_bytes = sum(f.stat().st_size for f in files)
    return {"files": len(files), "row_groups": row_groups, "input_size_mb": round(total_bytes / 2**20, 2)}

def to_pandas_result(result):
    if isinstance(result, pd.DataFrame): return result.copy()
    if isinstance(result, pl.DataFrame): return result.to_pandas()
    return pd.DataFrame(result)

def normalize_checksum_value(value):
    if isinstance(value, np.ndarray):        return json.dumps([normalize_checksum_value(v) for v in value.tolist()], sort_keys=True)
    if isinstance(value, (list, tuple)):     return json.dumps([normalize_checksum_value(v) for v in value], sort_keys=True)
    if pd.isna(value):                       return "<NA>"
    if isinstance(value, (np.bool_, bool)):  return "true" if bool(value) else "false"
    if isinstance(value, (np.integer, int)): return str(int(value))
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return str(int(value)) if value.is_integer() else f"{value:.6f}"
    if isinstance(value, (pd.Timestamp, np.datetime64)): return str(pd.Timestamp(value).isoformat())
    return str(value)

def result_checksum(result):
    pdf = to_pandas_result(result).copy()
    pdf.columns = [str(c) for c in pdf.columns]
    for col in pdf.columns: pdf[col] = pdf[col].map(normalize_checksum_value)
    sort_cols = list(pdf.columns)
    if sort_cols: pdf = pdf.sort_values(sort_cols).reset_index(drop=True)
    return hashlib.md5(pdf.to_csv(index=False, lineterminator="\n").encode("utf-8")).hexdigest()[:12]

benchmark_results = []

def run_with_peak_delta(fn, interval=0.01):
    process = psutil.Process(os.getpid())
    def rss_mb(): return process.memory_info().rss / 2**20
    baseline = rss_mb(); peak = [baseline]; running = [True]
    def sample_memory():
        while running[0]:
            peak[0] = max(peak[0], rss_mb()); time.sleep(interval)
    sampler = threading.Thread(target=sample_memory, daemon=True); sampler.start()
    try:
        result = fn(); peak[0] = max(peak[0], rss_mb())
        return max(peak[0] - baseline, 0.0), result
    finally:
        running[0] = False; sampler.join(timeout=1)

def benchmark_case(library_engine, mode, query_name, layout, fn, repeats=REPEATS, notes="", data_format="parquet"):
    times = []; result = None; peak_mem = None
    profile = layout_profile(layout, data_format=data_format)
    for i in range(repeats):
        gc.collect(); start = time.perf_counter()
        if i == 0: peak_mem, result = run_with_peak_delta(fn)
        else:      result = fn()
        times.append(time.perf_counter() - start)
    pdf = to_pandas_result(result)
    row = {
        "library_engine": library_engine, "mode": mode, "query_name": query_name,
        "data_format": data_format, "layout": layout, "rows": len(pdf),
        "median_time_s": round(float(np.median(times)), 4),
        "iqr_time_s": round(float(np.percentile(times, 75) - np.percentile(times, 25)), 4),
        "peak_memory_mb": round(float(peak_mem), 2),
        "input_size_mb": profile["input_size_mb"], "files": profile["files"], "row_groups": profile["row_groups"],
        "result_check": result_checksum(pdf),
        "notes": (notes + "; " if notes else "") + "RSS delta sampled in notebook process",
    }
    benchmark_results.append(row)
    return row

def show_results():
    return pd.DataFrame(benchmark_results).sort_values(["query_name", "layout", "library_engine", "mode"]).reset_index(drop=True)

print(f"benchmark harness ready -- {len(BENCHMARK_COLUMNS)} result columns; call benchmark_case(...) / show_results()")


benchmark harness ready -- 14 result columns; call benchmark_case(...) / show_results()


## Part 3: Student tasks

### Task 1: Design two benchmark queries

Create two queries of your own choice. They must test different behavior.

Your queries should cover at least two of the following classes:

- selective filter plus aggregation,
- high-cardinality group-by,
- top-k or sorting,
- list/tag explode,
- window or rolling computation,
- query that produces a large output,
- query sensitive to partitioned vs. unpartitioned layout,
- query sensitive to column pruning, predicate pushdown, or row-group pruning.

For each query, write a short hypothesis before you run it:

- what does this query test?
- which library/engine do you expect to perform best?
- which library/engine may use the most memory?
- which physical layout should help, if any?


In [14]:
# TODO: Define your two query specifications in prose or structured metadata.
# Do not start benchmarking before you can explain what each query is supposed to test.

QUERY_DESCRIPTIONS = """
Query 1: Selective filter + aggregation + top-k

Logical operation:
Filter incidents by selected Category and IncidentGrade, then group by CountryCode.
For every country, calculate:
- number of rows,
- number of distinct IncidentId values,
- number of non-null MitreTechniques values.
Finally, return the top 20 countries sorted by incident count.

What this query tests:
This query tests selective filtering, aggregation, top-k sorting, column pruning and predicate pushdown.
It should benefit from the optimized Parquet layout sorted by Category, IncidentGrade and CountryCode.

Hypothesis:
DuckDB and Polars lazy should perform best because they can push filters and projections into the Parquet scan.
Pandas may use more memory because it materializes the selected data eagerly.
The optimized Parquet layout should help more than the unpartitioned layout, because the data is sorted by the filter columns.


Query 2: MITRE technique explode + high-cardinality group-by

Logical operation:
Read the MitreTechniques column, remove nulls, split the semicolon-separated list of techniques,
explode it into individual technique rows, and group by technique.
Return the top 30 most frequent MITRE techniques.

What this query tests:
This query tests string processing, list/tag explode, high-cardinality group-by and sorting.
It is more CPU- and memory-sensitive than Query 1 because one input row may produce multiple output rows.

Hypothesis:
Polars should perform well because it has efficient string/list operations and parallel execution.
DuckDB should also perform well with SQL-based string splitting and unnesting.
Pandas may use the most memory because explode and string operations are materialized eagerly.
Physical Parquet layout should matter less here than in Query 1, because the query mainly depends on one string column and does not use the sorted filter keys strongly.
"""
print(QUERY_DESCRIPTIONS)


Query 1: Selective filter + aggregation + top-k

Logical operation:
Filter incidents by selected Category and IncidentGrade, then group by CountryCode.
For every country, calculate:
- number of rows,
- number of distinct IncidentId values,
- number of non-null MitreTechniques values.
Finally, return the top 20 countries sorted by incident count.

What this query tests:
This query tests selective filtering, aggregation, top-k sorting, column pruning and predicate pushdown.
It should benefit from the optimized Parquet layout sorted by Category, IncidentGrade and CountryCode.

Hypothesis:
DuckDB and Polars lazy should perform best because they can push filters and projections into the Parquet scan.
Pandas may use more memory because it materializes the selected data eagerly.
The optimized Parquet layout should help more than the unpartitioned layout, because the data is sorted by the filter columns.


Query 2: MITRE technique explode + high-cardinality group-by

Logical operation:
Read t

### Task 2: Benchmark local libraries/engines

Implement your two queries in:

- Pandas 3.0.2 with the default NumPy-backed output from `pd.read_parquet(path)`,
- Pandas 3.0.2 with `pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")`,
- Polars,
- DuckDB.

For Polars, benchmark at least:

- eager execution,
- lazy execution with default collection,
- lazy execution with streaming engine.


In [15]:
# TODO: Pandas implementations of your two queries.
# Implement both Pandas read variants:
# 1. default backend: pd.read_parquet(path)
# 2. PyArrow backend: pd.read_parquet(path, engine="pyarrow", dtype_backend="pyarrow")
#
# Report dtypes for both variants and compare runtime/memory.


# --- Helper configuration for Task 2 ---------------------------------------------

# Choose a non-empty Category + IncidentGrade pair automatically.
# This avoids hardcoding values that may not exist in the dataset.
con = duckdb.connect()

q1_params = con.execute(f"""
    WITH pairs AS (
        SELECT
            Category,
            IncidentGrade,
            COUNT(*) AS n
        FROM read_parquet('{EVENTS_PATH}')
        WHERE Category IS NOT NULL
          AND IncidentGrade IS NOT NULL
        GROUP BY Category, IncidentGrade
    ),
    total AS (
        SELECT COUNT(*) AS total_n
        FROM read_parquet('{EVENTS_PATH}')
    )
    SELECT Category, IncidentGrade, n
    FROM pairs, total
    WHERE n > total_n * 0.005
      AND n < total_n * 0.20
    ORDER BY n ASC
    LIMIT 1
""").fetchdf()

con.close()

Q1_CATEGORY = q1_params.loc[0, "Category"]
Q1_INCIDENT_GRADE = q1_params.loc[0, "IncidentGrade"]

print("Query 1 filter parameters:")
print("Q1_CATEGORY =", Q1_CATEGORY)
print("Q1_INCIDENT_GRADE =", Q1_INCIDENT_GRADE)
print(q1_params)


# --- Task 2: Pandas implementations ----------------------------------------------

def pandas_q1(path, dtype_backend=None):
    read_kwargs = {
        "columns": ["Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"]
    }

    if dtype_backend == "pyarrow":
        read_kwargs.update({
            "engine": "pyarrow",
            "dtype_backend": "pyarrow"
        })

    df = pd.read_parquet(path, **read_kwargs)

    filtered = df[
        (df["Category"] == Q1_CATEGORY) &
        (df["IncidentGrade"] == Q1_INCIDENT_GRADE)
    ]

    result = (
        filtered
        .groupby("CountryCode", dropna=False)
        .agg(
            rows=("IncidentId", "size"),
            distinct_incidents=("IncidentId", "nunique"),
            mitre_non_null=("MitreTechniques", "count")
        )
        .reset_index()
        .sort_values(["rows", "CountryCode"], ascending=[False, True])
        .head(20)
        .reset_index(drop=True)
    )

    return result


def pandas_q2(path, dtype_backend=None):
    read_kwargs = {
        "columns": ["MitreTechniques"]
    }

    if dtype_backend == "pyarrow":
        read_kwargs.update({
            "engine": "pyarrow",
            "dtype_backend": "pyarrow"
        })

    df = pd.read_parquet(path, **read_kwargs)

    s = (
        df["MitreTechniques"]
        .dropna()
        .astype("string")
        .str.split(";")
        .explode()
        .str.strip()
    )

    s = s[s.notna() & (s != "")]

    result = (
        s
        .value_counts()
        .rename_axis("MitreTechnique")
        .reset_index(name="rows")
        .sort_values(["rows", "MitreTechnique"], ascending=[False, True])
        .head(30)
        .reset_index(drop=True)
    )

    return result


# Report dtypes for both Pandas variants
pandas_default_sample = pd.read_parquet(
    parquet_source("unpartitioned"),
    columns=["Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"]
)

pandas_arrow_sample = pd.read_parquet(
    parquet_source("unpartitioned"),
    columns=["Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"],
    engine="pyarrow",
    dtype_backend="pyarrow"
)

print("\nPandas default dtypes:")
print(pandas_default_sample.dtypes)

print("\nPandas PyArrow backend dtypes:")
print(pandas_arrow_sample.dtypes)


# Benchmark Query 1
benchmark_case(
    "Pandas",
    "default",
    "Q1_filter_groupby_topk",
    "unpartitioned",
    lambda: pandas_q1(parquet_source("unpartitioned")),
    notes="pd.read_parquet default backend"
)

benchmark_case(
    "Pandas",
    "pyarrow_dtype_backend",
    "Q1_filter_groupby_topk",
    "unpartitioned",
    lambda: pandas_q1(parquet_source("unpartitioned"), dtype_backend="pyarrow"),
    notes='pd.read_parquet(engine="pyarrow", dtype_backend="pyarrow")'
)

# Benchmark Query 2
benchmark_case(
    "Pandas",
    "default",
    "Q2_mitre_explode_groupby",
    "unpartitioned",
    lambda: pandas_q2(parquet_source("unpartitioned")),
    notes="pd.read_parquet default backend"
)

benchmark_case(
    "Pandas",
    "pyarrow_dtype_backend",
    "Q2_mitre_explode_groupby",
    "unpartitioned",
    lambda: pandas_q2(parquet_source("unpartitioned"), dtype_backend="pyarrow"),
    notes='pd.read_parquet(engine="pyarrow", dtype_backend="pyarrow")'
)

Query 1 filter parameters:
Q1_CATEGORY = Persistence
Q1_INCIDENT_GRADE = BenignPositive
      Category   IncidentGrade      n
0  Persistence  BenignPositive  55973

Pandas default dtypes:
Category             str
IncidentGrade        str
CountryCode        int64
IncidentId         int64
MitreTechniques      str
dtype: object

Pandas PyArrow backend dtypes:
Category           large_string[pyarrow]
IncidentGrade      large_string[pyarrow]
CountryCode               int64[pyarrow]
IncidentId                int64[pyarrow]
MitreTechniques    large_string[pyarrow]
dtype: object


{'library_engine': 'Pandas',
 'mode': 'pyarrow_dtype_backend',
 'query_name': 'Q2_mitre_explode_groupby',
 'data_format': 'parquet',
 'layout': 'unpartitioned',
 'rows': 30,
 'median_time_s': 6.9821,
 'iqr_time_s': 4.4686,
 'peak_memory_mb': 1263.14,
 'input_size_mb': 299.24,
 'files': 1,
 'row_groups': 78,
 'result_check': 'af851554aaf7',
 'notes': 'pd.read_parquet(engine="pyarrow", dtype_backend="pyarrow"); RSS delta sampled in notebook process'}

In [16]:
# TODO: Polars implementations of your two queries.
# Required modes:
# - eager: read_parquet -> transformations
# - lazy default: scan_parquet -> transformations -> collect()
# - lazy streaming: scan_parquet -> transformations -> collect(engine="streaming")

# --- Task 2: Polars implementations ----------------------------------------------

def polars_q1_eager(path):
    return (
        pl.read_parquet(path, columns=["Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"])
        .filter(
            (pl.col("Category") == Q1_CATEGORY) &
            (pl.col("IncidentGrade") == Q1_INCIDENT_GRADE)
        )
        .group_by("CountryCode")
        .agg([
            pl.len().alias("rows"),
            pl.col("IncidentId").n_unique().alias("distinct_incidents"),
            pl.col("MitreTechniques").drop_nulls().len().alias("mitre_non_null")
        ])
        .sort(["rows", "CountryCode"], descending=[True, False])
        .head(20)
    )


def polars_q1_lazy(path, streaming=False):
    lf = (
        pl.scan_parquet(path)
        .select(["Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"])
        .filter(
            (pl.col("Category") == Q1_CATEGORY) &
            (pl.col("IncidentGrade") == Q1_INCIDENT_GRADE)
        )
        .group_by("CountryCode")
        .agg([
            pl.len().alias("rows"),
            pl.col("IncidentId").n_unique().alias("distinct_incidents"),
            pl.col("MitreTechniques").drop_nulls().len().alias("mitre_non_null")
        ])
        .sort(["rows", "CountryCode"], descending=[True, False])
        .head(20)
    )

    if streaming:
        return lf.collect(engine="streaming")
    return lf.collect()


def polars_q2_eager(path):
    return (
        pl.read_parquet(path, columns=["MitreTechniques"])
        .filter(pl.col("MitreTechniques").is_not_null())
        .with_columns(
            pl.col("MitreTechniques")
            .str.split(";")
            .alias("MitreTechnique")
        )
        .explode("MitreTechnique")
        .with_columns(
            pl.col("MitreTechnique").str.strip_chars()
        )
        .filter(pl.col("MitreTechnique") != "")
        .group_by("MitreTechnique")
        .agg(pl.len().alias("rows"))
        .sort(["rows", "MitreTechnique"], descending=[True, False])
        .head(30)
    )


def polars_q2_lazy(path, streaming=False):
    lf = (
        pl.scan_parquet(path)
        .select(["MitreTechniques"])
        .filter(pl.col("MitreTechniques").is_not_null())
        .with_columns(
            pl.col("MitreTechniques")
            .str.split(";")
            .alias("MitreTechnique")
        )
        .explode("MitreTechnique")
        .with_columns(
            pl.col("MitreTechnique").str.strip_chars()
        )
        .filter(pl.col("MitreTechnique") != "")
        .group_by("MitreTechnique")
        .agg(pl.len().alias("rows"))
        .sort(["rows", "MitreTechnique"], descending=[True, False])
        .head(30)
    )

    if streaming:
        return lf.collect(engine="streaming")
    return lf.collect()


# Query 1
benchmark_case(
    "Polars",
    "eager",
    "Q1_filter_groupby_topk",
    "unpartitioned",
    lambda: polars_q1_eager(parquet_source("unpartitioned")),
    notes="read_parquet eager"
)

benchmark_case(
    "Polars",
    "lazy",
    "Q1_filter_groupby_topk",
    "unpartitioned",
    lambda: polars_q1_lazy(parquet_source("unpartitioned")),
    notes="scan_parquet lazy collect"
)

benchmark_case(
    "Polars",
    "lazy_streaming",
    "Q1_filter_groupby_topk",
    "unpartitioned",
    lambda: polars_q1_lazy(parquet_source("unpartitioned"), streaming=True),
    notes='scan_parquet lazy collect(engine="streaming")'
)

# Query 2
benchmark_case(
    "Polars",
    "eager",
    "Q2_mitre_explode_groupby",
    "unpartitioned",
    lambda: polars_q2_eager(parquet_source("unpartitioned")),
    notes="read_parquet eager"
)

benchmark_case(
    "Polars",
    "lazy",
    "Q2_mitre_explode_groupby",
    "unpartitioned",
    lambda: polars_q2_lazy(parquet_source("unpartitioned")),
    notes="scan_parquet lazy collect"
)

benchmark_case(
    "Polars",
    "lazy_streaming",
    "Q2_mitre_explode_groupby",
    "unpartitioned",
    lambda: polars_q2_lazy(parquet_source("unpartitioned"), streaming=True),
    notes='scan_parquet lazy collect(engine="streaming")'
)

{'library_engine': 'Polars',
 'mode': 'lazy_streaming',
 'query_name': 'Q2_mitre_explode_groupby',
 'data_format': 'parquet',
 'layout': 'unpartitioned',
 'rows': 30,
 'median_time_s': 0.3387,
 'iqr_time_s': 0.0043,
 'peak_memory_mb': 1.61,
 'input_size_mb': 299.24,
 'files': 1,
 'row_groups': 78,
 'result_check': 'af851554aaf7',
 'notes': 'scan_parquet lazy collect(engine="streaming"); RSS delta sampled in notebook process'}

In [17]:
# TODO: DuckDB SQL implementations of your two queries.
# Consider querying Parquet files directly instead of first loading all data into Pandas.

# --- Task 2: DuckDB implementations ----------------------------------------------

def duckdb_q1(path):
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT
            CountryCode,
            COUNT(*) AS rows,
            COUNT(DISTINCT IncidentId) AS distinct_incidents,
            COUNT(MitreTechniques) AS mitre_non_null
        FROM read_parquet('{path}')
        WHERE Category = ?
          AND IncidentGrade = ?
        GROUP BY CountryCode
        ORDER BY rows DESC, CountryCode ASC
        LIMIT 20
    """, [Q1_CATEGORY, Q1_INCIDENT_GRADE]).fetchdf()
    con.close()
    return result


def duckdb_q2(path):
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT
            TRIM(technique) AS MitreTechnique,
            COUNT(*) AS rows
        FROM read_parquet('{path}'),
        UNNEST(string_split(MitreTechniques, ';')) AS t(technique)
        WHERE MitreTechniques IS NOT NULL
          AND TRIM(technique) <> ''
        GROUP BY MitreTechnique
        ORDER BY rows DESC, MitreTechnique ASC
        LIMIT 30
    """).fetchdf()
    con.close()
    return result


benchmark_case(
    "DuckDB",
    "sql",
    "Q1_filter_groupby_topk",
    "unpartitioned",
    lambda: duckdb_q1(parquet_source("unpartitioned")),
    notes="SQL directly over Parquet"
)

benchmark_case(
    "DuckDB",
    "sql",
    "Q2_mitre_explode_groupby",
    "unpartitioned",
    lambda: duckdb_q2(parquet_source("unpartitioned")),
    notes="SQL string_split + unnest directly over Parquet"
)

{'library_engine': 'DuckDB',
 'mode': 'sql',
 'query_name': 'Q2_mitre_explode_groupby',
 'data_format': 'parquet',
 'layout': 'unpartitioned',
 'rows': 30,
 'median_time_s': 0.6316,
 'iqr_time_s': 0.0187,
 'peak_memory_mb': 99.29,
 'input_size_mb': 299.24,
 'files': 1,
 'row_groups': 78,
 'result_check': 'af851554aaf7',
 'notes': 'SQL string_split + unnest directly over Parquet; RSS delta sampled in notebook process'}

In [18]:
show_results()

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,iqr_time_s,peak_memory_mb,input_size_mb,files,row_groups,result_check,notes
0,DuckDB,sql,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.4305,0.0787,60.91,299.24,1,78,e5697a8a09a1,SQL directly over Parquet; RSS delta sampled i...
1,Pandas,default,Q1_filter_groupby_topk,parquet,unpartitioned,2,1.4224,0.2739,929.25,299.24,1,78,e5697a8a09a1,pd.read_parquet default backend; RSS delta sam...
2,Pandas,pyarrow_dtype_backend,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.8614,0.1281,26.58,299.24,1,78,e5697a8a09a1,"pd.read_parquet(engine=""pyarrow"", dtype_backen..."
3,Polars,eager,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.2249,0.0187,672.79,299.24,1,78,e5697a8a09a1,read_parquet eager; RSS delta sampled in noteb...
4,Polars,lazy,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.1641,0.0058,1.03,299.24,1,78,e5697a8a09a1,scan_parquet lazy collect; RSS delta sampled i...
5,Polars,lazy_streaming,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.1721,0.0257,49.25,299.24,1,78,e5697a8a09a1,"scan_parquet lazy collect(engine=""streaming"");..."
6,DuckDB,sql,Q2_mitre_explode_groupby,parquet,unpartitioned,30,0.6316,0.0187,99.29,299.24,1,78,af851554aaf7,SQL string_split + unnest directly over Parque...
7,Pandas,default,Q2_mitre_explode_groupby,parquet,unpartitioned,30,6.8048,0.3125,1284.41,299.24,1,78,af851554aaf7,pd.read_parquet default backend; RSS delta sam...
8,Pandas,pyarrow_dtype_backend,Q2_mitre_explode_groupby,parquet,unpartitioned,30,6.9821,4.4686,1263.14,299.24,1,78,af851554aaf7,"pd.read_parquet(engine=""pyarrow"", dtype_backen..."
9,Polars,eager,Q2_mitre_explode_groupby,parquet,unpartitioned,30,1.4314,0.0212,732.81,299.24,1,78,af851554aaf7,read_parquet eager; RSS delta sampled in noteb...


### Task 2 results summary

The benchmark results confirm that all implementations produced equivalent results, because the `result_check` values are identical for each query across Pandas, Polars and DuckDB.

For Query 1, the fastest variant was Polars lazy execution with a median runtime of about 0.164 s and very low measured memory usage. DuckDB was also efficient, while Pandas default was the slowest and used the most memory. This confirms the hypothesis that engines with query optimization and Parquet predicate/projection pushdown perform better for selective filtering and aggregation.

For Query 2, Polars lazy streaming was clearly the fastest variant, with a median runtime of about 0.339 s and the lowest measured peak memory usage. DuckDB also performed well, while both Pandas variants were much slower and used over 1.2 GB of memory. This shows that string splitting, exploding and grouping are significantly more expensive in Pandas because the intermediate data is materialized eagerly.

Overall, Polars lazy/streaming and DuckDB were better suited for these analytical queries over Parquet files. Pandas was easier to use, but it required much more memory, especially for the MITRE technique explode query.

### Task 3: File format and Parquet layout optimization

Choose one of your two queries and test whether physical layout changes the amount of data read and the runtime.

Required comparison:

- default Parquet layout: randomly ordered data, one file or the default layout from your generator,
- optimized Parquet layout: choose a layout based on the query pattern, for example sorting by filter columns, changing `row_group_size`, partitioning by a selective column, or using writer-level pruning aids such as bloom filters if your writer and reader clearly support them,
- negative baseline: CSV or JSON/JSONL for the same query, to show what is lost without Parquet column pruning and predicate pushdown.

Use CSV if you do not have a strong reason to prefer JSON/JSONL. If your full dataset contains nested/list columns, create a flat query-specific CSV/JSON baseline containing only the columns needed by the selected query.

Report at least:

- file format and physical layout,
- total input size and number of files,
- runtime and peak memory,
- result checksum/equivalence,
- evidence of pruning where available: query plan, number of files read/skipped, row groups read/skipped, or a clear explanation if the engine does not expose these metrics.

Do not just create a faster layout accidentally. Explain why the layout should help this query.


In [20]:
# TODO 3: Build and benchmark one optimized layout for one selected query.
# Suggested steps:
# 1. Choose one query with a selective filter or column subset.
# 2. Write a baseline Parquet file/directory.
# 3. Write an optimized Parquet file/directory, e.g. sorted and with a selected row_group_size.
# 4. Write CSV or JSONL as a required negative baseline.
#    If your full dataset has nested/list columns, write a flat query-specific baseline with the columns needed by the selected query.
# 5. Benchmark the same logical query on default Parquet, optimized Parquet, and CSV/JSONL.
# 6. Record IO/pruning evidence where available.

# YOUR CODE HERE

# --- Task 3: File format and Parquet layout optimization --------------------------
# We use Query 1 because it has selective filters on Category and IncidentGrade.
# This makes it a good candidate for testing Parquet layout optimization.

def duckdb_q1_csv(path):
    con = duckdb.connect()
    result = con.execute(f"""
        SELECT
            CountryCode,
            COUNT(*) AS rows,
            COUNT(DISTINCT IncidentId) AS distinct_incidents,
            COUNT(MitreTechniques) AS mitre_non_null
        FROM read_csv_auto('{path}')
        WHERE Category = ?
          AND IncidentGrade = ?
        GROUP BY CountryCode
        ORDER BY rows DESC, CountryCode ASC
        LIMIT 20
    """, [Q1_CATEGORY, Q1_INCIDENT_GRADE]).fetchdf()
    con.close()
    return result


# Default Parquet layout
benchmark_case(
    "DuckDB",
    "sql_layout_default",
    "Q1_layout_filter_groupby_topk",
    "unpartitioned",
    lambda: duckdb_q1(parquet_source("unpartitioned")),
    notes="Task 3 default single-file Parquet layout"
)

# Optimized Parquet layout: sorted by Category, IncidentGrade, CountryCode
benchmark_case(
    "DuckDB",
    "sql_layout_optimized",
    "Q1_layout_filter_groupby_topk",
    "optimized",
    lambda: duckdb_q1(parquet_source("optimized")),
    notes="Task 3 optimized Parquet sorted by filter/group keys with smaller row groups"
)

# Partitioned Parquet layout
benchmark_case(
    "DuckDB",
    "sql_layout_partitioned",
    "Q1_layout_filter_groupby_topk",
    "partitioned",
    lambda: duckdb_q1(parquet_source("partitioned")),
    notes="Task 3 Hive-partitioned Parquet by ts_date_bucketed"
)

# Negative baseline: CSV with only query-specific columns
benchmark_case(
    "DuckDB",
    "sql_csv_baseline",
    "Q1_layout_filter_groupby_topk",
    "csv_baseline",
    lambda: duckdb_q1_csv(CSV_EVENTS_PATH),
    notes="Task 3 negative CSV baseline with only columns required by Query 1",
    data_format="csv"
)

task3_results = show_results()
task3_results[task3_results["query_name"] == "Q1_layout_filter_groupby_topk"]


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,iqr_time_s,peak_memory_mb,input_size_mb,files,row_groups,result_check,notes
6,DuckDB,sql_csv_baseline,Q1_layout_filter_groupby_topk,csv,csv_baseline,2,1.2569,0.0312,248.29,392.70,1,NaN,e5697a8a09a1,Task 3 negative CSV baseline with only columns...
7,DuckDB,sql_layout_optimized,Q1_layout_filter_groupby_topk,parquet,optimized,2,0.1228,0.0078,1.29,220.72,1,95.0,e5697a8a09a1,Task 3 optimized Parquet sorted by filter/grou...
8,DuckDB,sql_layout_partitioned,Q1_layout_filter_groupby_topk,parquet,partitioned,2,0.3622,0.0241,12.30,258.36,30,143.0,e5697a8a09a1,Task 3 Hive-partitioned Parquet by ts_date_buc...
9,DuckDB,sql_layout_default,Q1_layout_filter_groupby_topk,parquet,unpartitioned,2,0.3622,0.0114,2.73,299.24,1,78.0,e5697a8a09a1,Task 3 default single-file Parquet layout; RSS...


### Task 3 results summary

For Task 3, Query 1 was selected because it contains selective filters on `Category` and `IncidentGrade`. This makes it suitable for testing whether the physical Parquet layout can improve query performance through pruning and better data locality.

The optimized Parquet layout was the fastest variant. Its median runtime was about 0.123 s, compared with about 0.362 s for the default Parquet layout. The optimized file was also smaller than the default Parquet file, 220.72 MB compared with 299.24 MB. This confirms that sorting by the filter and grouping keys, together with smaller row groups, helped this query.

The partitioned Parquet layout did not improve runtime compared with the default layout. This is expected because the partitioning was based on `ts_date_bucketed`, while Query 1 filters by `Category` and `IncidentGrade`. Therefore, the partition column did not match the query predicate.

The CSV baseline was the slowest variant, with a median runtime of about 1.257 s and the largest input size, 392.70 MB. This shows what is lost without Parquet column pruning, metadata, compression and row-group-level optimization.

All variants produced the same `result_check`, so the layout and file format changes affected performance but not the correctness of the result.

### Task 4: Execution Modes & Analysis

**Goal**: deep dive into execution models, memory limits, and the decision boundary between single-node and distributed processing.

#### Lazy vs. eager vs. streaming

Use Polars to compare execution time and peak memory for the same logical operation in these modes:

- eager execution: `read_parquet` -> filter/transform,
- lazy execution: `scan_parquet` -> filter/transform -> `collect()`,
- streaming execution: `scan_parquet` -> filter/transform -> `collect(engine="streaming")`,
- streaming output: `scan_parquet` -> filter/transform -> `sink_parquet(...)`.

Important distinction:

- `collect(engine="streaming")` uses the streaming engine but still materializes the final result as a DataFrame.
- `sink_parquet(...)` or another sink writes the result to disk and is the better pattern when the output may be large.

Choose a query where this distinction matters. A tiny aggregate result may not show meaningful peak-memory differences. A better stress case keeps many rows, selects several columns, performs a non-trivial filter, and writes a large output.

**Run memory-sensitive variants in separate processes if possible.** If you run all modes in one notebook kernel, previous allocations and engine caches can hide the real memory difference. At minimum, call `gc.collect()` before each measured run and discuss the limitation.

If peak memory is almost identical across modes, increase the dataset size, increase the output size, measure each mode in a fresh process, or explain why your query is not memory-stressful enough.


In [21]:
# TODO 4: Implement Polars execution-mode experiments.
#
# Required variants:
# 1. eager: read_parquet -> filter/transform
# 2. lazy: scan_parquet -> filter/transform -> collect()
# 3. streaming collect: scan_parquet -> filter/transform -> collect(engine="streaming")
# 4. streaming sink: scan_parquet -> filter/transform -> sink_parquet(...)
#
# Recommended:
# - use a query whose output has many rows, not a tiny aggregate table,
# - measure each mode in a fresh process if possible,
# - call gc.collect() before each measured run,
# - record runtime, peak memory, output row count, and output size,
# - append results to benchmark_results.

# YOUR CODE HERE

# --- Task 4: Polars execution modes ----------------------------------------------
# This task compares eager execution, lazy collect, streaming collect and streaming sink.
# The query intentionally produces a relatively large output, so the difference between
# collecting into memory and writing directly to disk can be observed.

TASK4_OUTPUT_DIR = Path("task4_outputs")
TASK4_OUTPUT_DIR.mkdir(exist_ok=True)

def task4_large_output_eager(path):
    return (
        pl.read_parquet(
            path,
            columns=["Timestamp", "Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"]
        )
        .filter(pl.col("MitreTechniques").is_not_null())
        .with_columns([
            pl.col("Timestamp").str.slice(0, 10).alias("event_date"),
            pl.col("MitreTechniques").str.len_chars().alias("mitre_text_len")
        ])
        .select([
            "event_date",
            "Category",
            "IncidentGrade",
            "CountryCode",
            "IncidentId",
            "MitreTechniques",
            "mitre_text_len"
        ])
    )


def task4_large_output_lazy(path, streaming=False):
    lf = (
        pl.scan_parquet(path)
        .select(["Timestamp", "Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"])
        .filter(pl.col("MitreTechniques").is_not_null())
        .with_columns([
            pl.col("Timestamp").str.slice(0, 10).alias("event_date"),
            pl.col("MitreTechniques").str.len_chars().alias("mitre_text_len")
        ])
        .select([
            "event_date",
            "Category",
            "IncidentGrade",
            "CountryCode",
            "IncidentId",
            "MitreTechniques",
            "mitre_text_len"
        ])
    )

    if streaming:
        return lf.collect(engine="streaming")
    return lf.collect()


def task4_large_output_sink(path, output_path):
    lf = (
        pl.scan_parquet(path)
        .select(["Timestamp", "Category", "IncidentGrade", "CountryCode", "IncidentId", "MitreTechniques"])
        .filter(pl.col("MitreTechniques").is_not_null())
        .with_columns([
            pl.col("Timestamp").str.slice(0, 10).alias("event_date"),
            pl.col("MitreTechniques").str.len_chars().alias("mitre_text_len")
        ])
        .select([
            "event_date",
            "Category",
            "IncidentGrade",
            "CountryCode",
            "IncidentId",
            "MitreTechniques",
            "mitre_text_len"
        ])
    )

    if os.path.exists(output_path):
        os.remove(output_path)

    lf.sink_parquet(output_path, compression="zstd")

    output_size_mb = os.path.getsize(output_path) / 2**20
    output_rows = pl.scan_parquet(output_path).select(pl.len().alias("rows")).collect().item()

    return pd.DataFrame({
        "output_path": [output_path],
        "output_rows": [output_rows],
        "output_size_mb": [round(output_size_mb, 2)]
    })


# Eager execution: read_parquet materializes data immediately.
benchmark_case(
    "Polars",
    "task4_eager_collect",
    "Q3_large_output_filter_transform",
    "unpartitioned",
    lambda: task4_large_output_eager(parquet_source("unpartitioned")),
    notes="Task 4 eager read_parquet; large output materialized in memory"
)

# Lazy collect: optimizer can push projection/filter down, but final DataFrame is materialized.
benchmark_case(
    "Polars",
    "task4_lazy_collect",
    "Q3_large_output_filter_transform",
    "unpartitioned",
    lambda: task4_large_output_lazy(parquet_source("unpartitioned"), streaming=False),
    notes="Task 4 lazy scan_parquet collect; output materialized in memory"
)

# Streaming collect: uses streaming engine, but still returns a final DataFrame.
benchmark_case(
    "Polars",
    "task4_streaming_collect",
    "Q3_large_output_filter_transform",
    "unpartitioned",
    lambda: task4_large_output_lazy(parquet_source("unpartitioned"), streaming=True),
    notes='Task 4 collect(engine="streaming"); still materializes final DataFrame'
)

# Streaming sink: writes the result directly to disk instead of collecting the full result.
benchmark_case(
    "Polars",
    "task4_streaming_sink",
    "Q3_large_output_filter_transform",
    "unpartitioned",
    lambda: task4_large_output_sink(
        parquet_source("unpartitioned"),
        str(TASK4_OUTPUT_DIR / "large_output_streaming_sink.parquet")
    ),
    notes="Task 4 sink_parquet; writes output to disk instead of collecting full result"
)

task4_results = show_results()
task4_results[task4_results["query_name"] == "Q3_large_output_filter_transform"]


,library_engine,mode,query_name,data_format,layout,rows,median_time_s,iqr_time_s,peak_memory_mb,input_size_mb,files,row_groups,result_check,notes
16,Polars,task4_eager_collect,Q3_large_output_filter_transform,parquet,unpartitioned,4048451,0.4999,0.0937,709.27,299.24,1,78.0,6ff74d4a472f,Task 4 eager read_parquet; large output materi...
17,Polars,task4_lazy_collect,Q3_large_output_filter_transform,parquet,unpartitioned,4048451,0.3953,0.0392,32.34,299.24,1,78.0,6ff74d4a472f,Task 4 lazy scan_parquet collect; output mater...
18,Polars,task4_streaming_collect,Q3_large_output_filter_transform,parquet,unpartitioned,4048451,0.3675,0.0061,23.78,299.24,1,78.0,6ff74d4a472f,"Task 4 collect(engine=""streaming""); still mate..."
19,Polars,task4_streaming_sink,Q3_large_output_filter_transform,parquet,unpartitioned,1,0.9976,0.0446,48.15,299.24,1,78.0,23ae2f53944a,Task 4 sink_parquet; writes output to disk ins...


### Task 4 results summary

Task 4 compared four Polars execution modes on the same logical operation that produced a large output of more than 4 million rows.

The eager variant was the most memory-intensive collect-based mode. It used about 709 MB of peak memory and had a median runtime of about 0.500 s. This is expected because `read_parquet` loads data eagerly before applying the transformations.

Lazy execution improved both runtime and memory usage. The lazy collect variant reduced peak memory to about 32 MB and runtime to about 0.395 s. This shows the benefit of query optimization, projection pushdown and delayed execution.

Streaming collect was the fastest collect-based variant, with a median runtime of about 0.368 s and peak memory usage of about 24 MB. However, it still materialized the final result as a DataFrame, so it is not the same as writing the result directly to disk.

The streaming sink variant used `sink_parquet`, so the large result was written directly to disk. The benchmark row contains only one returned row because the function returns a small summary of the output file, not the full result. This explains why its checksum differs from the collect-based variants. The sink variant was slower in this run, about 0.998 s, because it included the cost of writing and validating the output file. However, it avoids keeping the full result in Python memory, which is important when the output is too large to materialize comfortably.

Overall, lazy and streaming execution modes were more memory-efficient than eager execution. `collect(engine="streaming")` is useful for reducing memory pressure during execution, while `sink_parquet` is the better pattern when the output is large and should be stored rather than loaded into memory.

In [22]:
show_results()

,library_engine,mode,query_name,data_format,layout,rows,median_time_s,iqr_time_s,peak_memory_mb,input_size_mb,files,row_groups,result_check,notes
0,DuckDB,sql,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.4305,0.0787,60.91,299.24,1,78.0,e5697a8a09a1,SQL directly over Parquet; RSS delta sampled i...
1,Pandas,default,Q1_filter_groupby_topk,parquet,unpartitioned,2,1.4224,0.2739,929.25,299.24,1,78.0,e5697a8a09a1,pd.read_parquet default backend; RSS delta sam...
2,Pandas,pyarrow_dtype_backend,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.8614,0.1281,26.58,299.24,1,78.0,e5697a8a09a1,"pd.read_parquet(engine=""pyarrow"", dtype_backen..."
3,Polars,eager,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.2249,0.0187,672.79,299.24,1,78.0,e5697a8a09a1,read_parquet eager; RSS delta sampled in noteb...
4,Polars,lazy,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.1641,0.0058,1.03,299.24,1,78.0,e5697a8a09a1,scan_parquet lazy collect; RSS delta sampled i...
5,Polars,lazy_streaming,Q1_filter_groupby_topk,parquet,unpartitioned,2,0.1721,0.0257,49.25,299.24,1,78.0,e5697a8a09a1,"scan_parquet lazy collect(engine=""streaming"");..."
6,DuckDB,sql_csv_baseline,Q1_layout_filter_groupby_topk,csv,csv_baseline,2,1.2569,0.0312,248.29,392.70,1,NaN,e5697a8a09a1,Task 3 negative CSV baseline with only columns...
7,DuckDB,sql_layout_optimized,Q1_layout_filter_groupby_topk,parquet,optimized,2,0.1228,0.0078,1.29,220.72,1,95.0,e5697a8a09a1,Task 3 optimized Parquet sorted by filter/grou...
8,DuckDB,sql_layout_partitioned,Q1_layout_filter_groupby_topk,parquet,partitioned,2,0.3622,0.0241,12.30,258.36,30,143.0,e5697a8a09a1,Task 3 Hive-partitioned Parquet by ts_date_buc...
9,DuckDB,sql_layout_default,Q1_layout_filter_groupby_topk,parquet,unpartitioned,2,0.3622,0.0114,2.73,299.24,1,78.0,e5697a8a09a1,Task 3 default single-file Parquet layout; RSS...


### Final benchmark summary

The final benchmark table combines the results from Task 2, Task 3 and Task 4. The `result_check` values confirm that equivalent query variants returned the same results. For Query 1, all implementations produced checksum `e5697a8a09a1`, and for Query 2, all implementations produced checksum `af851554aaf7`.

For the local engine comparison in Task 2, Polars lazy was the fastest and most memory-efficient option for Query 1, with a median runtime of about 0.164 s and only about 1 MB of measured peak memory. For Query 2, Polars lazy streaming was the best variant, with a median runtime of about 0.339 s and about 1.6 MB of measured peak memory. Pandas was clearly the most memory-intensive approach, especially for the MITRE explode query, where both Pandas variants used more than 1.2 GB of memory.

For the layout experiment in Task 3, the optimized Parquet layout was the best option. It reduced runtime from about 0.362 s for the default Parquet layout to about 0.123 s. The CSV baseline was the slowest and largest input format, which confirms the benefits of Parquet compression, column pruning and row-group metadata.

For the execution-mode experiment in Task 4, eager Polars execution used the most memory among the collect-based variants. Lazy collect and streaming collect were much more memory-efficient, while streaming collect was also the fastest collect-based mode. The streaming sink variant was slower because it included writing the output to disk, but it avoided returning the full 4-million-row result as a DataFrame.

Overall, the results show that query optimization, lazy execution, streaming execution and Parquet-aware physical layouts can significantly reduce both runtime and memory usage. Pandas is convenient, but for larger analytical workloads over Parquet files, Polars and DuckDB were more efficient in this benchmark.

### Task 5: Thread and core scalability

Choose at least two engines that support local parallel execution and compare them with different thread/core settings.

Suggested settings:

- DuckDB: configure number of threads for the connection.
- Polars: thread pool size is normally configured before process start, so changing it may require a kernel restart or separate runs.

In your report, do not only show speedup. Explain why scaling is or is not close to linear.

Hints:
- Polars' thread pool is built from `POLARS_MAX_THREADS` at `import polars` time; there is no runtime setter, so each thread-count must run in a fresh process/kernel.
- DuckDB threads, by contrast, can be changed on a live connection with `SET threads=N`.

In [ ]:
# TODO: Run selected scalability experiments and append results to benchmark_results.

## Final notebook report

The rendered notebook is your final submission. You do not submit a separate report.

Before submitting, make sure this notebook contains:

- full name,
- link to this notebook in your fork,
- two query descriptions with hypotheses,
- local benchmark table for Pandas 3.0.2 default backend, Pandas 3.0.2 PyArrow backend, Polars, DuckDB,
- file-format and Parquet-layout experiment with a required CSV/JSON negative baseline and evidence about column pruning, predicate pushdown, file pruning, or row-group pruning,
- Polars eager vs. lazy vs. streaming vs. sink discussion,
- local scalability results for selected libraries/engines,
- plots or tables that support your claims,
- final recommendations.

Do not commit generated data files, benchmark outputs, credentials, or local environment files.


### Final answers

Fill in the cells below. These answers should be visible in the rendered notebook.

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 1: Which query best exposes the difference between DataFrame and SQL engines?
FINAL_ANSWER_1 = """
TODO: Write your answer here.
"""
display_answer("Final answer 1", FINAL_ANSWER_1)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 2: Which query is most memory-sensitive?
FINAL_ANSWER_2 = """
TODO: Write your answer here. Refer to measured peak memory and dataset/query shape.
"""
display_answer("Final answer 2", FINAL_ANSWER_2)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 3: Did lazy execution change the amount of data read or materialized?
FINAL_ANSWER_3 = """
TODO: Write your answer here. Refer to predicate/projection pushdown or query plans if available.
"""
display_answer("Final answer 3", FINAL_ANSWER_3)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 4: Did streaming collection reduce memory, runtime, or both?
FINAL_ANSWER_4 = """
TODO: Write your answer here. Distinguish collect(engine="streaming") from sink_parquet(...).
"""
display_answer("Final answer 4", FINAL_ANSWER_4)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 5: When was a streaming sink more appropriate than collecting the result?
FINAL_ANSWER_5 = """
TODO: Write your answer here. Mention output size and whether the final result needed to be materialized in Python.
"""
display_answer("Final answer 5", FINAL_ANSWER_5)

In [ ]:
from IPython.display import Markdown, display

def display_answer(title, text):
    display(Markdown(f"**{title}**\n\n{text.strip()}"))

# TODO FINAL 6: How did Pandas default backend compare with the PyArrow dtype backend?
FINAL_ANSWER_6 = """
TODO: Write your answer here. Mention runtime, memory, dtypes, and whether string-heavy or IO-heavy queries changed the result.
"""
display_answer("Final answer 6", FINAL_ANSWER_6)
